# Moral Reasoner Data ETL

A notebook that **Extracts → Transforms → Loads**
**Moral Reasoner** dataset (`moral.data`) with pandas.

| Item | Details |
|------|------|
| Source | T.R. Shultz & J.M. Daley (UC Irvine, 1994) |
| Target | `guilty(case)` — guilty (1) / not guilty (0) |
| Instances | 202 cases (102 guilty / 100 not guilty) |
| Auxiliary relations | 23 (y/n, numeric, categorical) |
| Raw format | Prolog facts (Horn clauses) |

**Workflow**

1. Parse Prolog facts in `moral.data` with regex (Extract)
2. Build a per-case feature table, convert `y/n` → 1/0 etc. (Transform)
3. Save CSV/Parquet and verify round-trip (Load)
4. Bonus: reproduce the inference rules of `moral.theory` in pandas to verify label agreement


## 1. Environment setup


In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

print("pandas :", pd.__version__)
print("numpy  :", np.__version__)


pandas : 2.3.3
numpy  : 2.3.5


In [2]:
DATA_DIR = Path.cwd()
DATA_PATH = DATA_DIR / "moral.data"
if not DATA_PATH.exists():
    DATA_PATH = Path(r"C:\Users\USER\OneDrive\Desktop\[16-08-26] Cloudy\00.Moral Reasoner") / "moral.data"

OUT_DIR = DATA_PATH.parent / "output"
OUT_DIR.mkdir(exist_ok=True)

print("Data file  :", DATA_PATH)
print("Output dir :", OUT_DIR)
print("Exists     :", DATA_PATH.exists())


Data file  : C:\Users\USER\Desktop\[16-08-26] Cloudy\00.Moral Reasoner\moral.data
Output dir : C:\Users\USER\Desktop\[16-08-26] Cloudy\00.Moral Reasoner\output
Exists     : True


## 2. Extract — read the raw file & parse Prolog facts

`moral.data` groups facts into blocks, one per case:

```
guilty(p0).              ← label (guilty)
not(guilty(n0)).         ← label (not guilty)
produce_harm(p0,y).      ← feature fact
severity_harm(p0,1).
...
```


In [3]:
raw_text = DATA_PATH.read_text(encoding="utf-8")
raw_lines = raw_text.splitlines()

print("Total lines       :", len(raw_lines))
print("Non-empty lines   :", sum(1 for l in raw_lines if l.strip()))
print("File size (bytes) :", DATA_PATH.stat().st_size)

print("\n--- First 12 lines preview ---")
for line in raw_lines[:12]:
    print(line)


Total lines       : 5049
Non-empty lines   : 4848
File size (bytes) : 122546

--- First 12 lines preview ---
guilty(p0).
sufficient_for_harm(p0,n).
produce_harm(p0,y).
plan_known(p0,y).
plan_include_harm(p0,y).
someone_else_cause_harm(p0,n).
outrank_perpetrator(p0,n).
monitor(p0,y).
harm_caused_as_planned(p0,y).
goal_outweigh_harm(p0,y).
goal_achieveable_less_harmful(p0,n).
foresee_intervention(p0,y).


In [4]:
LABEL_RE = re.compile(r"^(not\()?guilty\((\w+)\)\)?\.\s*$")
FACT_RE  = re.compile(r"^(\w+)\((\w+)\s*,\s*(\w+)\)\.\s*$")


def parse_moral_data(text: str) -> tuple[list, list]:
    """Parse Prolog facts in moral.data into per-case record dicts.

    - Case start: `guilty(pX).`       → guilty = 1 (guilty)
    - Case start: `not(guilty(nX)).`  → guilty = 0 (not guilty)
    - Other lines: `predicate(caseID, value).` auxiliary facts

    Returns (records, skipped), where skipped collects (line, reason)
    tuples for every non-empty line that was not parsed, so that
    silent data loss is impossible.
    """
    records: list = []
    skipped: list = []
    current = None

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        m_label = LABEL_RE.match(line)
        if m_label:
            if current is not None:
                records.append(current)
            current = {
                "case_id": m_label.group(2),
                "guilty": 0 if m_label.group(1) else 1,
            }
            continue

        m_fact = FACT_RE.match(line)
        if m_fact:
            if current is not None:
                current[m_fact.group(1)] = m_fact.group(3)
            else:
                skipped.append((raw_line, "fact before first label"))
            continue

        skipped.append((raw_line, "no matching pattern"))

    if current is not None:
        records.append(current)
    return records, skipped


In [5]:
records, skipped = parse_moral_data(raw_text)

print("Parsed cases  :", len(records))
print("Duplicate IDs :", len(records) - len({r["case_id"] for r in records}))
print("Skipped lines :", len(skipped))
for line, reason in skipped[:10]:
    print(f"  [{reason}] {line.strip()}")
if len(skipped) > 10:
    print(f"  ... ({len(skipped) - 10} more)")
assert not skipped, f"{len(skipped)} non-empty lines were not parsed"

print("\n--- Sample record (p0) ---")
print(json.dumps(records[0], indent=2, ensure_ascii=False))


Parsed cases  : 202
Duplicate IDs : 0
Skipped lines : 0

--- Sample record (p0) ---
{
  "case_id": "p0",
  "guilty": 1,
  "sufficient_for_harm": "n",
  "produce_harm": "y",
  "plan_known": "y",
  "plan_include_harm": "y",
  "someone_else_cause_harm": "n",
  "outrank_perpetrator": "n",
  "monitor": "y",
  "harm_caused_as_planned": "y",
  "goal_outweigh_harm": "y",
  "goal_achieveable_less_harmful": "n",
  "foresee_intervention": "y",
  "external_cause": "n",
  "control_perpetrator": "y",
  "benefit_protagonist": "y",
  "careful": "y",
  "benefit_victim": "0",
  "severity_harm": "1",
  "achieve_goal": "n",
  "intervening_contribution": "n",
  "foreseeability": "high",
  "external_force": "n",
  "mental_state": "negligent",
  "necessary_for_harm": "y"
}


In [6]:
AUX_FEATURES = [
    "sufficient_for_harm", "produce_harm", "plan_known", "plan_include_harm",
    "someone_else_cause_harm", "outrank_perpetrator", "monitor",
    "harm_caused_as_planned", "goal_outweigh_harm", "goal_achieveable_less_harmful",
    "foresee_intervention", "external_cause", "control_perpetrator",
    "benefit_protagonist", "careful", "benefit_victim", "severity_harm",
    "achieve_goal", "intervening_contribution", "foreseeability",
    "external_force", "mental_state", "necessary_for_harm",
]

missing_count = {
    f: sum(1 for r in records if f not in r)
    for f in AUX_FEATURES
}
print("Features :", len(AUX_FEATURES))
print("Missing cases per feature :", missing_count)
print("Features per case (min/max) :",
      min(len(r) - 2 for r in records), "/",
      max(len(r) - 2 for r in records))


Features : 23
Missing cases per feature : {'sufficient_for_harm': 0, 'produce_harm': 0, 'plan_known': 0, 'plan_include_harm': 0, 'someone_else_cause_harm': 0, 'outrank_perpetrator': 0, 'monitor': 0, 'harm_caused_as_planned': 0, 'goal_outweigh_harm': 0, 'goal_achieveable_less_harmful': 0, 'foresee_intervention': 0, 'external_cause': 0, 'control_perpetrator': 0, 'benefit_protagonist': 0, 'careful': 0, 'benefit_victim': 0, 'severity_harm': 0, 'achieve_goal': 0, 'intervening_contribution': 0, 'foreseeability': 0, 'external_force': 0, 'mental_state': 0, 'necessary_for_harm': 0}
Features per case (min/max) : 23 / 23


## 3. Transform — dataframe conversion

Conversion rules:

| Raw value | Converted |
|---------|------|
| `y` / `n` | 1 / 0 (int8) |
| `foreseeability`: n / low / high | 0 / 1 / 2 (ordinal) |
| `mental_state`: neither / negligent / reckless / intend | 0 / 1 / 2 / 3 (ordinal) |
| `benefit_victim`, `severity_harm` | integer (int) |

Categorical columns holding the raw values (`foreseeability`, `mental_state`)
are kept alongside the numeric codes.


In [7]:
BINARY_FEATURES = [
    "sufficient_for_harm", "produce_harm", "plan_known", "plan_include_harm",
    "someone_else_cause_harm", "outrank_perpetrator", "monitor",
    "harm_caused_as_planned", "goal_outweigh_harm", "goal_achieveable_less_harmful",
    "foresee_intervention", "external_cause", "control_perpetrator",
    "benefit_protagonist", "careful", "achieve_goal",
    "intervening_contribution", "external_force", "necessary_for_harm",
]
NUMERIC_FEATURES = ["benefit_victim", "severity_harm"]

YES_NO_MAP         = {"y": 1, "n": 0}
FORESEEABILITY_MAP = {"n": 0, "low": 1, "high": 2}
MENTAL_STATE_MAP   = {"neither": 0, "negligent": 1, "reckless": 2, "intend": 3}

df = pd.DataFrame.from_records(records).set_index("case_id")

df["guilty"] = df["guilty"].astype("int8")

for col in BINARY_FEATURES:
    df[col] = df[col].map(YES_NO_MAP).astype("int8")

for col in NUMERIC_FEATURES:
    df[col] = df[col].astype(int)

df["foreseeability"] = df["foreseeability"].astype("category")
df["mental_state"]    = df["mental_state"].astype("category")

df["foreseeability_code"] = df["foreseeability"].map(FORESEEABILITY_MAP).astype("int8")
df["mental_state_code"]    = df["mental_state"].map(MENTAL_STATE_MAP).astype("int8")

print("Transform complete:", df.shape)


Transform complete: (202, 26)


In [8]:
COL_ORDER = ["guilty", *AUX_FEATURES, "foreseeability_code", "mental_state_code"]
df = df[COL_ORDER]

print(df.head(8).to_string())
print("\n--- dtypes ---")
print(df.dtypes.to_string())
print("\n--- Numeric summary ---")
print(df[["guilty", "benefit_victim", "severity_harm",
          "foreseeability_code", "mental_state_code"]].describe().to_string())


         guilty  sufficient_for_harm  produce_harm  plan_known  plan_include_harm  someone_else_cause_harm  outrank_perpetrator  monitor  harm_caused_as_planned  goal_outweigh_harm  goal_achieveable_less_harmful  foresee_intervention  external_cause  control_perpetrator  benefit_protagonist  careful  benefit_victim  severity_harm  achieve_goal  intervening_contribution foreseeability  external_force mental_state  necessary_for_harm  foreseeability_code  mental_state_code
case_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    
p0            1                    0            

In [9]:
print("Total cases :", len(df))
print("Label distribution :")
print(df["guilty"].value_counts().sort_index().to_string())

for col in ["foreseeability", "mental_state"]:
    print(f"\n{col} distribution :")
    print(df[col].value_counts().to_string())

assert len(df) == 202
assert df["guilty"].sum() == 102
assert df.index.is_unique
assert df.isna().sum().sum() == 0, "Missing values found"

print("\nBasic checks passed : 202 rows / guilty 102 / not guilty 100 / missing 0")


Total cases : 202
Label distribution :
guilty
0    100
1    102

foreseeability distribution :
foreseeability
low     104
high     96
n         2

mental_state distribution :
mental_state
reckless     65
neither      63
negligent    62
intend       12

Basic checks passed : 202 rows / guilty 102 / not guilty 100 / missing 0


## 4. Load — save & round-trip verification


In [10]:
CSV_PATH = OUT_DIR / "moral_cases.csv"
df.to_csv(CSV_PATH, encoding="utf-8-sig", index=True)
print("CSV saved      :", CSV_PATH)

try:
    PARQUET_PATH = OUT_DIR / "moral_cases.parquet"
    df.to_parquet(PARQUET_PATH)
    print("Parquet saved  :", PARQUET_PATH)
except ImportError:
    print("pyarrow not installed → skipping Parquet (pip install pyarrow)")

# Round-trip verification
loaded = pd.read_csv(CSV_PATH, index_col="case_id")
assert loaded.shape == df.shape
assert list(loaded.columns) == list(df.columns)
assert loaded.index.equals(df.index)
for col in df.columns:
    assert loaded[col].astype(str).equals(df[col].astype(str)), f"Column mismatch: {col}"
print("Round-trip verification passed :", loaded.shape)


CSV saved      : C:\Users\USER\Desktop\[16-08-26] Cloudy\00.Moral Reasoner\output\moral_cases.csv
Parquet saved  : C:\Users\USER\Desktop\[16-08-26] Cloudy\00.Moral Reasoner\output\moral_cases.parquet
Round-trip verification passed : (202, 26)


## 5. Bonus — reproduce the moral.theory inference logic

`moral.theory` defines `guilty/1` with these Horn-clause rules:

- `guilty(X) :- blameworthy(X)` or `vicarious_blame(X)`
- `blameworthy(X) :- responsible(X), not(justified(X)), severity_harm > benefit_victim`
- `responsible(X) :- cause, not(accident), voluntary, foreseeable, not(intervening_cause)`
- `accident` / `reckless` / `negligent` / `intend` are derived from the
  `mental_state` facts plus secondary rules

Prolog's `not/1` is **negation as failure**, so in this fully-fact dataset
it can be reproduced as `not(P) = P is False`.

> Note: the original theory uses the predicate name `reponsible` (a typo).


In [11]:
d = df.copy()

# --- cause ---
d["cause"] = (
    (d["produce_harm"] == 1)
    | (d["necessary_for_harm"] == 1)
    | (d["sufficient_for_harm"] == 1)
)

# --- intend / reckless / negligent ---
d["strong_intend"] = (d["mental_state"] == "intend") | (
    (d["plan_known"] == 1)
    & (d["plan_include_harm"] == 1)
    & (d["harm_caused_as_planned"] == 1)
)

d["discount_intent"] = d["external_cause"] == 1
d["weak_intend1"] = (~d["discount_intent"]) | (d["monitor"] == 1) | (d["benefit_protagonist"] == 1)

d["reckless"] = (d["mental_state"] == "reckless") | (
    (d["careful"] == 0) & (~d["strong_intend"]) & (d["foreseeability"] == "high")
)
d["negligent"] = (d["mental_state"] == "negligent") | (
    (d["careful"] == 0) & (~d["strong_intend"]) & (d["foreseeability"] == "low")
)

d["weak_intend"] = d["weak_intend1"] & (~d["reckless"]) & (~d["negligent"])
d["intend"] = d["strong_intend"] | d["weak_intend"]

# --- accident / foreseeable / voluntary / intervening_cause ---
d["accident"] = (~d["intend"]) & (~d["reckless"]) & (~d["negligent"])
d["foreseeable"] = d["foreseeability"].isin(["high", "low"])
d["voluntary"] = d["external_force"] == 0
d["intervening_cause"] = (d["intervening_contribution"] == 1) & (d["foresee_intervention"] == 0)

# --- responsible ---
d["responsible"] = (
    d["cause"]
    & (~d["accident"])
    & d["voluntary"]
    & d["foreseeable"]
    & (~d["intervening_cause"])
)

# --- justified / vicarious ---
d["justified"] = (
    (d["achieve_goal"] == 1)
    & (d["goal_outweigh_harm"] == 1)
    & (d["goal_achieveable_less_harmful"] == 0)
)
d["vicarious"] = (
    (d["someone_else_cause_harm"] == 1)
    & (d["outrank_perpetrator"] == 1)
    & (d["control_perpetrator"] == 1)
)

# --- blameworthy / vicarious_blame / guilty ---
sev_gt_ben = d["severity_harm"] > d["benefit_victim"]
d["blameworthy"] = d["responsible"] & (~d["justified"]) & sev_gt_ben
d["vicarious_blame"] = d["vicarious"] & (~d["justified"]) & sev_gt_ben
d["guilty_pred"] = d["blameworthy"] | d["vicarious_blame"]

print("Theory-based prediction columns created")


Theory-based prediction columns created


In [12]:
n_correct = (d["guilty_pred"] == d["guilty"]).sum()
acc = n_correct / len(d)
print("Label agreement : {:.2%} ({}/{})".format(acc, n_correct, len(d)))

mismatch = d[d["guilty_pred"] != d["guilty"]]
if len(mismatch) == 0:
    print("Theory predictions match moral.data labels 100%.")
else:
    print("Mismatched cases:")
    print(mismatch[["guilty", "guilty_pred"]].to_string())

print("\nIntermediate concept frequencies:")
intermediate = [
    "cause", "accident", "reckless", "negligent", "intend",
    "foreseeable", "voluntary", "intervening_cause", "responsible",
    "justified", "vicarious", "blameworthy", "vicarious_blame",
]
print(d[intermediate].sum().to_string())

THEORY_PATH = OUT_DIR / "moral_cases_with_theory.csv"
d.to_csv(THEORY_PATH, encoding="utf-8-sig")
print("\nTheory-column data saved :", THEORY_PATH)


Label agreement : 100.00% (202/202)
Theory predictions match moral.data labels 100%.

Intermediate concept frequencies:
cause                188
accident               2
reckless              92
negligent             90
intend                71
foreseeable          200
voluntary            148
intervening_cause     28
responsible          127
justified             14
vicarious             24
blameworthy           96
vicarious_blame       17

Theory-column data saved : C:\Users\USER\Desktop\[16-08-26] Cloudy\00.Moral Reasoner\output\moral_cases_with_theory.csv


Authors: T.R. Shultz, J.M. Daley
